# コイン参照コードの使用例：各機能の確認

## 準備

In [1]:
var { api } = await import('../lib/load-blockchain-api.mjs');
var { loadWallet } = await import('../lib/load-wallet.mjs');
var { adminWallet, rpc, deploySmartContract } = await import('../lib/notebook-util.mjs');

動作確認に使用するユーザをロードします。

In [2]:
var users = [];
for(var i=0; i<=5; i++){
    var wfstr = await loadWallet(`wallet-user${i}.json`);
    var wallet = await api.unlockWalletFile(await api.parseWalletFile(wfstr), '_paswrd_');
    var id = (await rpc.call(wallet, 'c1query', {type: 'search', key: 'me'})).value[0].id;
    users[i] = {id, wallet};
    console.log(`user${i}:`, id, wallet.address);
}

user0: u73621973 eQiA3bsfZunQ8KsRpkTgMGndwu46rq
user1: u28169743 epGyEm4BXFxFvLmXkDdDvcqspESPFu
user2: u53371386 eaZMspgRpi68EdReTZEiUkdJuKsdct
user3: u58185320 e5fVm5R4eKYXa7U57pshS4UQwuvGsp
user4: u10676071 eEYRycGm4znXxnf7bTngX72JmYk57r
user5: u00571239 eWW6LCfRSpoVuudzZHa4ZwCjkkizov


## コインのデプロイ

コインコントラクトのサンプルコードを読み出します。

In [3]:
var {default: func} = await import('./coin100.mjs');
var m = /function.*\{([\s\S]*)\}/.exec(func.toString());
var code = m[1];

コインの名称と略号を書き換えます。

In [4]:
var code = code.replace(/COIN_NAME = '.*'/, `COIN_NAME = 'sample COIN #1234'`);
var code = code.replace(/COIN_SYMBOL = '.*'/, `COIN_SYMBOL = 'COIN1234'`);

コインの発行枚数を1000とし、初期オーナをユーザ0に書き換えます。  
初期状態ではユーザ0が全コインのオーナになります。

In [5]:
var code = code.replace(/TOTAL_SUPPLY = \d+/, `TOTAL_SUPPLY = 1000`);
var code = code.replace(/INITIAL_OWNER = '.*'/, `INITIAL_OWNER = '${users[0].id}'`);

スマートコントラクトをデプロイします。  
変数coinidにコインコントラクトのIDが格納されます。

In [6]:
var func = new Function(code);
var coinid = await deploySmartContract({func:'string', args:'any'}, func, {name: 'coin100'});

コインコントラクトのアクセス範囲を、動作確認に使用するユーザに設定します。

In [7]:
await rpc.call(adminWallet, 'c1update', {id:coinid, prop:'accessible_to', value: users.map(e => e.id)});

{
  txno: 702086,
  txid: 'xHYdUn6ExAJs8Hi44NXaTYJAWcLDefuiUhpHAgFHbZyaz',
  status: 'ok',
  value: null
}

## コインコントラクトの各インタフェースの動作確認

### name
コインの名称を取得します。

In [8]:
var resp = await rpc.call(users[5].wallet, coinid, {func: 'name'});
console.log(resp);

{
  txno: 702087,
  txid: 'xPfiWV4tiKWLXbuvwCtNZHPsQmcYPpfMkMBeP7ahBHT9q',
  status: 'ok',
  value: 'sample COIN #1234'
}


### symbol
コインの略号を取得します。

In [9]:
var resp = await rpc.call(users[5].wallet, coinid, {func: 'symbol'});
console.log(resp);

{
  txno: 702088,
  txid: 'xwx3pt3BL8vUWZRzJcM7aUFRbmn3JvEwWCFQgoLny9r24',
  status: 'ok',
  value: 'COIN1234'
}


### totalSupply
コインの総発行枚数を取得します。

In [10]:
var resp = await rpc.call(users[5].wallet, coinid, {func: 'totalSupply'});
console.log(resp);

{
  txno: 702089,
  txid: 'xrJUmCoNxACwWShhMyQM6722GdcbjUNYtwBUgFu7fWnuPB',
  status: 'ok',
  value: 1000
}


### balanceOf
指定したオーナが現在保持するコインの数を取得します。

In [11]:
var resp = await rpc.call(users[5].wallet, coinid, {func: 'balanceOf', args: {owner: users[0].id}});
console.log(resp);

{
  txno: 702090,
  txid: 'xaQzkmfwNAJVwpiVW7zVKiJGDRDeRhkw5sRXiSoaiG3Qx',
  status: 'ok',
  value: 1000
}


In [12]:
var resp = await rpc.call(users[5].wallet, coinid, {func: 'balanceOf', args: {owner: users[1].id}});
console.log(resp);

{
  txno: 702091,
  txid: 'xsWpnMLNxvkzfepvcLEdD5VLMGLm7tB63rhv4txApErkBB',
  status: 'ok',
  value: 0
}


### transfer
自身が保持するコインのうち、指定数量のコインを別のオーナに転送します。  
10コインをuser0からuser1へ転送します。

In [13]:
var resp = await rpc.call(users[0].wallet, coinid, {func: "transfer", args: {to: users[1].id, value: 10}});
console.log(resp);

{
  txno: 702092,
  txid: 'xPmhPExwuLcsDJCswVtAdjzZLEjWQ6NFct7tSpQThBms8',
  status: 'ok',
  value: true
}


転送後の保持数を確認します。

In [14]:
var resp = await rpc.call(users[5].wallet, coinid, {func: 'balanceOf', args: {owner: users[0].id}});
console.log('balanceOf(user0):', resp.value);
var resp = await rpc.call(users[5].wallet, coinid, {func: 'balanceOf', args: {owner: users[1].id}});
console.log('balanceOf(user1):', resp.value);

balanceOf(user0): 990
balanceOf(user1): 10


保持している数量を超えて転送しようとするとエラーになります。

In [15]:
var resp = await rpc.call(users[1].wallet, coinid, {func: "transfer", args: {to: users[2].id, value: 11}});
console.log(resp);

{
  txno: 702095,
  txid: 'xb7UzVDdBy6zxgurunVB3NveccV2fXFVYqaqH7hUchPUJB',
  status: 'thrown',
  value: 'insufficient balance'
}


### approve
自身が保持するコインについて、コインを転送する権限を委任します。

コイン20枚について、user0からの転送の権限を、user2に委任します。

In [16]:
var resp = await rpc.call(users[0].wallet, coinid, {func: 'approve', args: {spender: users[2].id, value: 20}});
console.log(resp);

{
  txno: 702096,
  txid: 'xKyqt4p2N5yeNuN2Cp7AbPXZt2avgDBz5EVXWZjo299Sr',
  status: 'ok',
  value: true
}


### allowance
転送権限の委任数量を取得します。

In [17]:
var resp = await rpc.call(users[5].wallet, coinid, {func: 'allowance', args: {owner: users[0].id, spender: users[2].id}});
console.log(resp);

{
  txno: 702097,
  txid: 'xyqogztwykqsquNdWPo8bNLo7RfUkJJT5LWYwb2JmQLVz',
  status: 'ok',
  value: 20
}


委任されていない場合は、0が返ります。

In [18]:
var resp = await rpc.call(users[5].wallet, coinid, {func: 'allowance', args: {owner: users[1].id, spender: users[2].id}});
console.log(resp);

{
  txno: 702098,
  txid: 'x4aoEHyFfef8Vh8oDXcBNiWC8iMa6PdEuj2yQZ3AS5idz',
  status: 'ok',
  value: 0
}


現在の保持数を確認しておきます。

In [19]:
for(var i=0; i<=3; i++){var resp = await rpc.call(users[5].wallet, coinid, {func: 'balanceOf', args: {owner: users[i].id}});
console.log(`balanceOf(user${i}):`, resp.value);
}

balanceOf(user0): 990
balanceOf(user1): 10
balanceOf(user2): 0
balanceOf(user3): 0


### approveの効果
転送権限の委任により、委任先の操作によるコインの転送が可能になります。  
user0からuser3への5コインの転送をuser2が実行します。

In [20]:
var resp = await rpc.call(users[2].wallet, coinid, {func: "transferFrom",
    args: {
      from: users[0].id,
      to: users[3].id,
      value: 5
    }
});
console.log(resp);

{
  txno: 702103,
  txid: 'xxMTX38ujAJVuPhRUGvDwjavLjhv26qq3ZdxaH79fkAnEB',
  status: 'ok',
  value: true
}


転送後のコイン数量を確認します。

In [21]:
for(var i=0; i<=3; i++){var resp = await rpc.call(users[5].wallet, coinid, {func: 'balanceOf', args: {owner: users[i].id}});
console.log(`balanceOf(user${i}):`, resp.value);
}

balanceOf(user0): 985
balanceOf(user1): 10
balanceOf(user2): 0
balanceOf(user3): 5


転送後は、転送権限の委任の数量は減っています。

In [22]:
var resp = await rpc.call(users[5].wallet, coinid, {func: 'allowance', args: {owner: users[0].id, spender: users[2].id}});
console.log(resp);

{
  txno: 702108,
  txid: 'xJQ7bsNBxEBBhCEtDAxgVaRoiZe8QkWh5RsMVCXjPx4MGB',
  status: 'ok',
  value: 15
}


委任の数量を超えて転送しようとすると、エラーとなります。

In [23]:
var resp = await rpc.call(users[2].wallet, coinid, {func: "transferFrom",
    args: {
      from: users[0].id,
      to: users[3].id,
      value: 16
    }
});
console.log(resp);

{
  txno: 702109,
  txid: 'xHf6aYonJZMCTvzn8y7bzan99MDytQU2gKqPwVARr9TPJB',
  status: 'thrown',
  value: 'insufficient allowance'
}


### 委任の取り消し
委任を取り消すには、approveを使って委任量を0に上書きします。

In [24]:
var resp = await rpc.call(users[0].wallet, coinid, {func: 'approve', args: {spender: users[2].id, value: 0}});
console.log(resp);

{
  txno: 702110,
  txid: 'x3K2DkUamSizmThNYpHnURDsSAHc299dbxLNEU6MejFzEB',
  status: 'ok',
  value: true
}


転送権限の委任数量が0になっていることを確認します。

In [25]:
var resp = await rpc.call(users[5].wallet, coinid, {func: 'allowance', args: {owner: users[0].id, spender: users[2].id}});
console.log(resp);

{
  txno: 702111,
  txid: 'xYZqnqYL2V6xrjFhGjiqBKr22jDhkC9SPuxPfQFmGuPvp',
  status: 'ok',
  value: 0
}


委任が取り消されたため、コインの転送はできなくなります。

In [26]:
var resp = await rpc.call(users[2].wallet, coinid, {func: "transferFrom",
    args: {
      from: users[0].id,
      to: users[3].id,
      value: 1
    }
});
console.log(resp);

{
  txno: 702112,
  txid: 'xcS6423Xvdb8dRnsjQYwAnqAdguxc2ByBH5PxzmVzMDuDB',
  status: 'thrown',
  value: 'insufficient allowance'
}


### burn
burnはインタフェースとしては実装していませんが、   
存在しないコントラクトID、例えばc0000に転送することで、コインを破棄する効果が期待できます。

In [27]:
var resp = await rpc.call(users[1].wallet, coinid, {func: "transfer", args: {to: 'c0000', value: 1}});
console.log(resp);

{
  txno: 702113,
  txid: 'x2SiYN2Dzg2XefmEuTVLqxLnjnMmEPWbyuY3KGL4SW3FAB',
  status: 'ok',
  value: true
}
